# HDB Resale Price — Stacking Ensemble v15c (Colab GPU)

**Changes from v15b → v15c:**
- ✅ `N_FOLDS=10` (back to 10-fold — 5-fold cost $85 in stack quality)
- ✅ CatBoost `lr=0.02` + `iterations=20000` (v15b hit ceiling at i=9999 every fold)
- ✅ XGBoost `lr=0.01` + `n_estimators=10000` (kept — showed −$31 improvement)
- ✅ LGB `num_leaves=255` + `min_data_in_leaf=20` (kept from v15b)
- ⚠️ LGB stays on CPU (LGB GPU blocked by `street_name` having 486 unique categories)

**Architecture:** 3 base models → RidgeCV meta-learner
- **LightGBM** — native categoricals, CPU, `lr=0.03`, `leaves=255`
- **XGBoost** — ordinal-encoded cats, **GPU** (`device='cuda'`), `lr=0.01`
- **CatBoost** — ordered-TE cats, **GPU** (`task_type='GPU'`), `lr=0.02`

**Runtime estimate on Colab T4:**
- LGB per fold: ~400–500s (CPU, 255 leaves)
- XGB per fold: ~50–70s (GPU, lr=0.01, ~5500 iters)
- CatBoost per fold: ~200–350s (GPU, lr=0.02, ~10000 iters)
- **Total 10-fold: ~70–90 min**

**Version history:**
- v15 : 10-fold, lr=0.03 all, leaves=127 → stack **$21,354** ✅ best
- v15b: 5-fold, lr=0.01 XGB/CAT, leaves=255 → stack $21,439 ❌ worse
- v15c: 10-fold, lr=0.01 XGB / lr=0.02 CAT, leaves=255 → **target <$21,354**

> Make sure Runtime → Change runtime type → **GPU (T4)** before running.

In [ ]:
# Install / upgrade packages
!pip install -q lightgbm xgboost catboost --upgrade

# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU detected: {result.stdout.strip()}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

In [ ]:
import warnings, time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
warnings.filterwarnings('ignore')
print('All imports OK')
print(f'LGB {lgb.__version__}  XGB {xgb.__version__}')

## Data Upload

**Option A — Google Drive (recommended for large files):**
```python
from google.colab import drive
drive.mount('/content/drive')
# Then set DATA_DIR to your Drive folder, e.g.:
# DATA_DIR = '/content/drive/MyDrive/hdb/'
```

**Option B — Direct upload** (run the cell below, click Choose Files, select all 3 CSVs).

In [ ]:
# ── Option A: Mount Google Drive ──────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/hdb/'   # <-- change to your folder

# ── Option B: Direct upload ───────────────────────────────────────────────
from google.colab import files
print('Select train.csv, test.csv, sample_sub_reg.csv')
uploaded = files.upload()   # choose all 3 files
DATA_DIR = '/content/'      # files land here after upload

import os
for f in ['train.csv', 'test.csv', 'sample_sub_reg.csv']:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        print(f'  {f}: OK ({os.path.getsize(path)//1024} KB)')
    else:
        print(f'  {f}: MISSING — please upload it')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
TRAIN   = DATA_DIR + 'train.csv'
TEST    = DATA_DIR + 'test.csv'
SAMPLE  = DATA_DIR + 'sample_sub_reg.csv'
SUB     = '/content/submission_stacking_v15c.csv'
SEED    = 42
N_FOLDS = 10   # back to 10-fold (5-fold cost $85 in stack quality in v15b)

In [ ]:
# ── Load & deduplicate ────────────────────────────────────────────────────
train = pd.read_csv(TRAIN, low_memory=False)
test  = pd.read_csv(TEST,  low_memory=False)
print(f'train {train.shape}  test {test.shape}')

before = len(train)
train = train.drop_duplicates(
    subset=[c for c in train.columns if c != 'id']).reset_index(drop=True)
print(f'dedup: {before} -> {len(train)} rows')

# Bool columns
bool_cols = ['commercial', 'market_hawker', 'multistorey_carpark', 'precinct_pavilion']
for c in bool_cols:
    for df in (train, test):
        df[c] = df[c].astype(str).str.upper().map(
            {'Y': 1, 'N': 0, 'TRUE': 1, 'FALSE': 0}).fillna(0).astype(np.int8)

within_cols = ['Mall_Within_500m', 'Mall_Within_1km', 'Mall_Within_2km',
               'Hawker_Within_500m', 'Hawker_Within_1km', 'Hawker_Within_2km']
for c in within_cols:
    for df in (train, test):
        df[c] = df[c].fillna(0)

mall_max = max(train['Mall_Nearest_Distance'].max(),
               test['Mall_Nearest_Distance'].max())
for df in (train, test):
    df['Mall_Nearest_Distance'] = df['Mall_Nearest_Distance'].fillna(mall_max)

print('Preprocessing done')

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────
COORDS = {
    'cbd'       : (1.2835, 103.8510),
    'orchard'   : (1.3048, 103.8318),
    'marina_bay': (1.2800, 103.8590),
    'jurong_e'  : (1.3329, 103.7436),
    'tampines'  : (1.3540, 103.9440),
    'woodlands' : (1.4382, 103.7891),
    'serangoon' : (1.3496, 103.8729),
}

for df in (train, test):
    ym = df['Tranc_YearMonth'].str.split('-', expand=True)
    yr = ym[0].astype(int); mo = ym[1].astype(int)
    df['tranc_year']     = yr
    df['tranc_month']    = mo
    df['tranc_time_idx'] = (yr - 2000) * 12 + mo
    df['month_sin']      = np.sin(2 * np.pi * mo / 12)
    df['month_cos']      = np.cos(2 * np.pi * mo / 12)

    df['age_at_sale']     = yr - df['lease_commence_date']
    df['remaining_lease'] = 99 - df['age_at_sale']
    df['cpf_eligible']    = (df['remaining_lease'] >= 60).astype(np.int8)
    df['lease_above_cpf'] = np.maximum(0, df['remaining_lease'] - 60)
    df['lease_bucket']    = pd.cut(df['remaining_lease'],
                                   bins=[0, 50, 60, 70, 80, 90, 99],
                                   labels=['<50','50-60','60-70','70-80','80-90','>90']).astype(str)
    df['storey_rel'] = df['mid_storey'] / (df['max_floor_lvl'] + 1)
    df['high_floor'] = (df['mid_storey'] >= 20).astype(np.int8)

    lat = df['Latitude']; lon = df['Longitude']
    for name, (clat, clon) in COORDS.items():
        df[f'dist_{name}'] = np.sqrt((lat - clat)**2 + (lon - clon)**2)
    centre_dists = [f'dist_{n}' for n in COORDS]
    df['dist_nearest_centre']      = df[centre_dists].min(axis=1)
    df['dist_nearest_centre_name'] = df[centre_dists].idxmin(axis=1).str.replace('dist_', '')

    df['mrt_quality_score']  = (df['mrt_interchange'] * 2 + df['bus_interchange']).astype(np.int8)
    df['mrt_walk_cat']       = pd.cut(df['mrt_nearest_distance'],
                                      bins=[0, 300, 500, 1000, np.inf],
                                      labels=['<300m','300-500m','500m-1km','>1km']).astype(str)
    df['effective_mrt_dist'] = df['mrt_nearest_distance'] / (1 + df['mrt_interchange'] * 0.5)
    df['transport_score']    = (df['mrt_interchange'] * 3 + df['bus_interchange'] +
                                (df['mrt_nearest_distance'] < 500).astype(int))

    df['sec_sch_tier']         = pd.cut(df['cutoff_point'], bins=[0, 200, 215, 230, 260],
                                        labels=['standard','good','very_good','elite']).astype(str)
    df['vacancy_score']        = 1.0 / (df['vacancy'] + 1)
    df['pri_popular']          = (df['vacancy'] < 40).astype(np.int8)
    df['pri_quality_dist']     = df['pri_sch_affiliation'] / (df['pri_sch_nearest_distance'] + 1) * 1000
    df['sec_quality_dist']     = df['cutoff_point'] / (df['sec_sch_nearest_dist'] + 1) * 1000
    df['pri_within_1km']       = (df['pri_sch_nearest_distance'] < 1000).astype(np.int8)
    df['pri_within_1km_aff']   = (df['pri_within_1km'] & df['pri_sch_affiliation']).astype(np.int8)
    df['sec_within_1km']       = (df['sec_sch_nearest_dist'] < 1000).astype(np.int8)
    df['sec_elite_within_2km'] = ((df['sec_sch_nearest_dist'] < 2000) & (df['cutoff_point'] >= 230)).astype(np.int8)
    df['pri_sch_dist_cbd']     = np.sqrt((df['pri_sch_latitude'] - 1.2835)**2 + (df['pri_sch_longitude'] - 103.851)**2)
    df['sec_sch_dist_cbd']     = np.sqrt((df['sec_sch_latitude'] - 1.2835)**2 + (df['sec_sch_longitude'] - 103.851)**2)

    df['hawker_quality'] = df['hawker_food_stalls'] / (df['Hawker_Nearest_Distance'] + 1)
    df['hawker_density'] = df['Hawker_Within_500m'] * 3 + df['Hawker_Within_1km']
    df['mall_density']   = df['Mall_Within_500m'] * 3 + df['Mall_Within_1km'] * 2 + df['Mall_Within_2km']
    df['amenity_score']  = df['mall_density'] + df['hawker_density']

    total = df['total_dwelling_units'].replace(0, 1)
    df['pct_3room']   = df['3room_sold']  / total
    df['pct_4room']   = df['4room_sold']  / total
    df['pct_5room']   = df['5room_sold']  / total
    df['pct_exec']    = df['exec_sold']   / total
    df['pct_rental']  = (df['1room_rental'] + df['2room_rental'] +
                         df['3room_rental'] + df['other_room_rental']) / total
    df['pct_premium'] = (df['5room_sold'] + df['exec_sold'] + df['multigen_sold']) / total

    df['area_x_storey']     = df['floor_area_sqm'] * df['mid_storey']
    df['area_x_age']        = df['floor_area_sqm'] * df['age_at_sale']
    df['mrt_x_interchange'] = df['mrt_nearest_distance'] * (1 - df['mrt_interchange'] * 0.4)
    df['lease_x_storey']    = df['remaining_lease'] * df['storey_rel']
    df['postal_sector']     = df['postal'].astype(str).str.zfill(6).str[:2]
    df['mrt_dist_cbd']      = np.sqrt((df['mrt_latitude'] - 1.2835)**2 + (df['mrt_longitude'] - 103.851)**2)

print('Feature engineering done')

In [ ]:
# ── Feature matrices: LGB / XGB / CatBoost ────────────────────────────────
DROP = ['id', 'resale_price', 'Tranc_YearMonth', 'address', 'postal',
        'block', 'residential', 'bus_stop_name']

CAT  = ['town', 'flat_type', 'street_name', 'storey_range', 'flat_model',
        'full_flat_type', 'planning_area', 'mrt_name', 'pri_sch_name',
        'sec_sch_name', 'postal_sector', 'lease_bucket', 'mrt_walk_cat',
        'sec_sch_tier', 'dist_nearest_centre_name']

for c in CAT:
    train[c] = train[c].astype(str)
    test[c]  = test[c].astype(str)

feature_cols = [c for c in train.columns if c not in DROP]
NUM = [c for c in feature_cols if c not in CAT]

y_raw    = train['resale_price'].values.astype(float)
y        = np.log1p(y_raw)
test_ids = test['id'].values
print(f'features: {len(feature_cols)}  ({len(NUM)} numeric + {len(CAT)} categorical)')

# LightGBM — native category dtype (CPU)
X_lgb      = train[feature_cols].copy()
X_test_lgb = test[feature_cols].copy()
for c in CAT:
    X_lgb[c]      = X_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')
    unified = pd.api.types.union_categoricals([X_lgb[c], X_test_lgb[c]]).categories
    X_lgb[c]      = X_lgb[c].cat.set_categories(unified)
    X_test_lgb[c] = X_test_lgb[c].cat.set_categories(unified)
for c in NUM:
    X_lgb[c]      = pd.to_numeric(X_lgb[c], errors='coerce')
    X_test_lgb[c] = pd.to_numeric(X_test_lgb[c], errors='coerce')

# XGBoost — ordinal-encoded, median-filled (GPU)
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_xgb      = train[feature_cols].copy()
X_test_xgb = test[feature_cols].copy()
for c in NUM:
    X_xgb[c]      = pd.to_numeric(X_xgb[c], errors='coerce')
    X_test_xgb[c] = pd.to_numeric(X_test_xgb[c], errors='coerce')
    med = X_xgb[c].median()
    X_xgb[c]      = X_xgb[c].fillna(med)
    X_test_xgb[c] = X_test_xgb[c].fillna(med)
X_xgb[CAT]      = enc.fit_transform(X_xgb[CAT].astype(str))
X_test_xgb[CAT] = enc.transform(X_test_xgb[CAT].astype(str))

# CatBoost — string categoricals, median-filled numerics (GPU)
X_cat      = train[feature_cols].copy()
X_test_cat = test[feature_cols].copy()
for c in CAT:
    X_cat[c]      = X_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)
for c in NUM:
    X_cat[c]      = pd.to_numeric(X_cat[c], errors='coerce')
    X_test_cat[c] = pd.to_numeric(X_test_cat[c], errors='coerce')
    med = X_cat[c].median()
    X_cat[c]      = X_cat[c].fillna(med)
    X_test_cat[c] = X_test_cat[c].fillna(med)

print('Feature matrices ready')

In [ ]:
# ── Model parameters ──────────────────────────────────────────────────────

# LightGBM — CPU (LGB GPU blocked: street_name has 486 unique cats > GPU limit of 255)
lgb_params = dict(
    objective='regression', metric='rmse',
    learning_rate=0.03,
    num_leaves=255,          # ✅ fixed from v15 (was 127)
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    min_data_in_leaf=20,     # ✅ fixed from v15 (was 30)
    lambda_l2=1.0, lambda_l1=0.1,
    max_cat_threshold=64, verbose=-1,
    n_estimators=10000,
    random_state=SEED,
    device='cpu',
)

# XGBoost — GPU, lr=0.01 (showed improvement in v15b: −$31 vs v15)
xgb_params = dict(
    objective='reg:squarederror', eval_metric='rmse',
    learning_rate=0.01,      # ✅ kept from v15b — showed improvement
    max_depth=8,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    n_estimators=10000,
    tree_method='hist', device='cuda',
    early_stopping_rounds=200, random_state=SEED,
)

# CatBoost — GPU, lr=0.02 (v15b used lr=0.01 but hit i=9999 ceiling every fold)
# lr=0.02 converges in ~10000 iters instead of ~30000, iterations=20000 gives headroom
cat_params = dict(
    iterations=20000,        # ✅ increased — v15b hit 9999 ceiling at lr=0.01
    learning_rate=0.02,      # ✅ compromise: faster than 0.01, better than 0.03
    depth=8,
    l2_leaf_reg=3,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=SEED,
    od_type='Iter',
    od_wait=200,
    task_type='GPU',
    devices='0',
    verbose=0,
    allow_writing_files=False,
)

print('Parameters set:')
print(f'  LGB : lr=0.03  leaves=255  n_est=10000   device=cpu')
print(f'  XGB : lr=0.01  depth=8     n_est=10000   device=cuda')
print(f'  CAT : lr=0.02  depth=8     iter=20000    task_type=GPU')

In [ ]:
# ── Level-0: 5-fold CV ────────────────────────────────────────────────────
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

lgb_oof   = np.zeros(len(train))
xgb_oof   = np.zeros(len(train))
cat_oof   = np.zeros(len(train))
lgb_preds = np.zeros(len(test))
xgb_preds = np.zeros(len(test))
cat_preds = np.zeros(len(test))

print(f'=== Level-0: Training {N_FOLDS}-fold CV ===\n')
t_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_lgb)):
    t0 = time.time()
    y_tr, y_va = y[tr_idx], y[va_idx]

    # LightGBM (GPU, lr=0.01, leaves=255)
    lgb_m = lgb.LGBMRegressor(**lgb_params)
    lgb_m.fit(X_lgb.iloc[tr_idx], y_tr,
              eval_set=[(X_lgb.iloc[va_idx], y_va)],
              categorical_feature=CAT,
              callbacks=[lgb.early_stopping(200, verbose=False),
                         lgb.log_evaluation(0)])
    lgb_oof[va_idx] = lgb_m.predict(X_lgb.iloc[va_idx], num_iteration=lgb_m.best_iteration_)
    lgb_preds      += lgb_m.predict(X_test_lgb, num_iteration=lgb_m.best_iteration_) / N_FOLDS
    lgb_best_iter   = lgb_m.best_iteration_

    # XGBoost (GPU, lr=0.01)
    xgb_m = xgb.XGBRegressor(**xgb_params)
    xgb_m.fit(X_xgb.iloc[tr_idx], y_tr,
              eval_set=[(X_xgb.iloc[va_idx], y_va)],
              verbose=False)
    xgb_oof[va_idx] = xgb_m.predict(X_xgb.iloc[va_idx])
    xgb_preds      += xgb_m.predict(X_test_xgb) / N_FOLDS
    xgb_best_iter   = xgb_m.best_iteration

    # CatBoost (GPU, lr=0.01)
    cat_m = CatBoostRegressor(**cat_params)
    cat_m.fit(X_cat.iloc[tr_idx], y_tr,
              cat_features=CAT,
              eval_set=(X_cat.iloc[va_idx], y_va),
              use_best_model=True)
    cat_oof[va_idx] = cat_m.predict(X_cat.iloc[va_idx])
    cat_preds      += cat_m.predict(X_test_cat) / N_FOLDS
    cat_best_iter   = cat_m.best_iteration_

    lgb_f = np.sqrt(np.mean((np.expm1(lgb_oof[va_idx]) - y_raw[va_idx])**2))
    xgb_f = np.sqrt(np.mean((np.expm1(xgb_oof[va_idx]) - y_raw[va_idx])**2))
    cat_f = np.sqrt(np.mean((np.expm1(cat_oof[va_idx]) - y_raw[va_idx])**2))
    elapsed = time.time() - t0
    print(f'  Fold {fold}: LGB={lgb_f:,.0f}(i={lgb_best_iter})  '
          f'XGB={xgb_f:,.0f}(i={xgb_best_iter})  '
          f'CAT={cat_f:,.0f}(i={cat_best_iter})  ({elapsed:.0f}s)')

lgb_rmse_oof = np.sqrt(np.mean((np.expm1(lgb_oof) - y_raw)**2))
xgb_rmse_oof = np.sqrt(np.mean((np.expm1(xgb_oof) - y_raw)**2))
cat_rmse_oof = np.sqrt(np.mean((np.expm1(cat_oof) - y_raw)**2))
print(f'\n  LGB OOF RMSE = ${lgb_rmse_oof:,.0f}')
print(f'  XGB OOF RMSE = ${xgb_rmse_oof:,.0f}')
print(f'  CAT OOF RMSE = ${cat_rmse_oof:,.0f}')
print(f'  Base models done in {(time.time()-t_start)/60:.1f} min')

In [ ]:
# ── Level-1: RidgeCV meta-learner ─────────────────────────────────────────
META_RAW = ['floor_area_sqm', 'mid_storey', 'remaining_lease',
            'dist_cbd', 'tranc_time_idx', 'pri_sch_nearest_distance',
            'mrt_nearest_distance', 'cutoff_point', 'pri_sch_dist_cbd']

def get_meta_raw(df, ref_df=None):
    cols = []
    for f in META_RAW:
        vals = df[f].copy()
        if vals.dtype == object:
            vals = pd.to_numeric(vals, errors='coerce')
        med = (ref_df[f] if ref_df is not None else vals).median()
        cols.append(vals.fillna(med).values)
    return np.column_stack(cols)

raw_tr = get_meta_raw(train)
raw_te = get_meta_raw(test, ref_df=train)

dis_lx_tr = lgb_oof - xgb_oof;  dis_lx_te = lgb_preds - xgb_preds
dis_lc_tr = lgb_oof - cat_oof;  dis_lc_te = lgb_preds - cat_preds
dis_xc_tr = xgb_oof - cat_oof;  dis_xc_te = xgb_preds - cat_preds

X_meta_tr = np.column_stack([lgb_oof, xgb_oof, cat_oof, dis_lx_tr, dis_lc_tr, dis_xc_tr, raw_tr])
X_meta_te = np.column_stack([lgb_preds, xgb_preds, cat_preds, dis_lx_te, dis_lc_te, dis_xc_te, raw_te])

scaler = StandardScaler()
X_meta_tr_sc = scaler.fit_transform(X_meta_tr)
X_meta_te_sc = scaler.transform(X_meta_te)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
meta = RidgeCV(alphas=alphas, cv=5)
meta.fit(X_meta_tr_sc, y)
print(f'Best alpha: {meta.alpha_}')

feat_names = ['lgb_oof', 'xgb_oof', 'cat_oof', 'dis_lx', 'dis_lc', 'dis_xc'] + META_RAW
print('Coefficients:')
for name, coef in zip(feat_names, meta.coef_):
    print(f'  {name:<28s}  {coef:+.4f}')

In [ ]:
# ── Evaluate + generate submission ────────────────────────────────────────
meta_oof_log = meta.predict(X_meta_tr_sc)
meta_oof_raw = np.expm1(meta_oof_log)

mse  = mean_squared_error(y_raw, meta_oof_raw)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_raw, meta_oof_raw)
r2   = r2_score(y_raw, meta_oof_raw)

# 3-way inverse-RMSE blend for comparison
w_l = 1/lgb_rmse_oof; w_x = 1/xgb_rmse_oof; w_c = 1/cat_rmse_oof
w_s = w_l + w_x + w_c
blend_oof  = (w_l*lgb_oof + w_x*xgb_oof + w_c*cat_oof) / w_s
blend_rmse = np.sqrt(np.mean((np.expm1(blend_oof) - y_raw)**2))

print('======== Stacking OOF Results (v15c) ========')
print(f'{"LightGBM  (Level-0)":<30s}  RMSE = ${lgb_rmse_oof:>10,.0f}')
print(f'{"XGBoost   (Level-0)":<30s}  RMSE = ${xgb_rmse_oof:>10,.0f}')
print(f'{"CatBoost  (Level-0)":<30s}  RMSE = ${cat_rmse_oof:>10,.0f}')
print(f'{"3-way blend":<30s}  RMSE = ${blend_rmse:>10,.0f}')
print(f'{"RidgeCV Stack (v15c)":<30s}  RMSE = ${rmse:>10,.0f}  <-- submit this')
print()
print(f'RMSE  = ${rmse:,.2f}')
print(f'MAE   = ${mae:,.2f}')
print(f'R2    = {r2:.6f}')
print(f'RMSE% = {rmse/y_raw.mean()*100:.2f}% of mean price')
print(f'vs v15  ($21,354):  {"BETTER" if rmse < 21354 else "WORSE"} by ${abs(rmse-21354):,.0f}')
print(f'vs v15b ($21,439):  {"BETTER" if rmse < 21439 else "WORSE"} by ${abs(rmse-21439):,.0f}')
print(f'Improvement over blend: ${blend_rmse-rmse:,.0f}')
print(f'Total time: {(time.time()-t_start)/60:.1f} min')

# Generate submission
meta_test_log = meta.predict(X_meta_te_sc)
pred_raw      = np.expm1(meta_test_log)

all_preds = pd.DataFrame({'Id': test_ids, 'Predicted': pred_raw})
sample    = pd.read_csv(SAMPLE)
sub       = sample[['Id']].merge(all_preds, on='Id', how='left')

assert len(sub) == len(sample), f'Row mismatch: {len(sub)} vs {len(sample)}'
assert sub['Predicted'].isna().sum() == 0, 'Missing predictions!'

sub.to_csv(SUB, index=False)
print(f'\nwrote {SUB}  ({len(sub)} rows)')
print(sub['Predicted'].describe())

In [ ]:
# ── Download submission CSV ───────────────────────────────────────────────
from google.colab import files
files.download(SUB)
print('Download started!')